# U03_C1 — Inside a Sequence Record — The NCBI Data Model
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 3:** The NCBI Data Model
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L5–L6 (NCBI Data Model, ASN.1, Seq-id, Bioseq)
**Before this:** U02_C1

## 1. What you will learn
- **Map** Biopython's `SeqRecord` attributes to the NCBI Abstract Syntax Notation One (ASN.1) data model components (`Seq-id`, `Bioseq`, `Seq-descr`, `Seq-annot`, `Seq-feat`) [LO1, LO2].
- **Navigate** the `record.annotations` dictionary to extract taxonomic and publication metadata (`Seq-descr`) [LO2].
- **Iterate** over `record.features` to inspect sequence annotations (`Seq-feat`), coordinates, and qualifiers [LO2].
- **Understand** how structured data models allow complex queries and relational database linkages [LO1].


## 2. Biological question
A biological sequence without metadata is just a string of letters. In clinical and computational genetics, knowing *what* the sequence is, *where* it came from (organism, chromosome, tissue), *which* parts code for protein, and *who* published the evidence is essential. NCBI designed a formal data specification called the **NCBI Data Model** (implemented in ASN.1 and XML). How does Biopython's `SeqRecord` represent this model, and how do we programmatically navigate its layers on human *HBB*?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U03_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: The NCBI Data Model Architecture
### Theory
To store diverse biological data without format inconsistencies, NCBI developed an object-oriented specification structured into five core concepts:
1. **`Bioseq`**: The fundamental biological sequence unit (a sequence object with length, alphabet, and topology).
2. **`Seq-id`**: The unambiguous identifier of the sequence (e.g. `NM_000518.5`, GI, or accession).
3. **`Seq-descr`** (Descriptors): Metadata applying to the entire sequence (organism, taxonomy, title, references, comments).
4. **`Seq-annot`** & **`Seq-feat`** (Features): Annotations placed at specific sequence intervals (e.g. `gene`, `CDS`, `exon`, `variation`).
5. **`Bioseq-set`**: Collections of related Bioseqs (e.g. mRNA + protein translated pair, or population phylogenetic set).

In Biopython, loading a GenBank record creates a `SeqRecord` that maps directly onto this hierarchy:
- `record.id` $\leftrightarrow$ `Seq-id`
- `record.seq` $\leftrightarrow$ `Bioseq` raw sequence
- `record.description` & `record.annotations` $\leftrightarrow$ `Seq-descr`
- `record.features` $\leftrightarrow$ `Seq-annot` (list of `SeqFeature` objects)


In [ ]:
# Worked Example: Loading HBB and mapping top-level NCBI Data Model elements
from Bio import SeqIO

gb_path = DATA_DIR / "genbank" / "hbb_human.gb"
record = SeqIO.read(gb_path, "genbank")

print("=== NCBI Data Model: Top-level Mapping ===")
print("Seq-id (record.id):         ", record.id)
print("Seq-id (record.name):       ", record.name)
print("Bioseq Length (len):        ", len(record.seq), "bp")
print("Bioseq Alphabet / Molecule: ", record.annotations.get("molecule_type", "mRNA"))
print("Seq-descr (record.description):")
print("  ", record.description)


## 4. Concept 2: Exploring Descriptors (Seq-descr)
### Theory
Descriptors (`Seq-descr`) provide contextual biological provenance. In Biopython, these are stored in `record.annotations` as a Python dictionary. Key descriptors include:
- `taxonomy`: Hierarchical lineage list (e.g. `['Eukaryota', 'Metazoa', 'Chordata', ...]`)
- `organism`: Scientific species name (e.g. `'Homo sapiens'`)
- `references`: List of literature citations (`Reference` objects with authors, title, journal, pubmed_id)
- `comment`: Curatorial comments from NCBI staff


In [ ]:
# Worked Example: Extracting Seq-descr elements from record.annotations
annotations = record.annotations

print("Organism:", annotations.get("organism"))
print("Taxonomic Lineage:")
lineage = annotations.get("taxonomy", [])
print(" -> ".join(lineage[:5]) + " -> ... -> " + lineage[-1])

print(f"Number of literature citations (Seq-descr References): {len(annotations.get('references', []))}")
first_ref = annotations["references"][0]
print("Primary Reference:")
print("  Title:  ", first_ref.title)
print("  Authors:", first_ref.authors)
print("  Journal:", first_ref.journal)
print("  PubMed: ", first_ref.pubmed_id)


## 5. Concept 3: Features and Annotations (Seq-annot & Seq-feat)
### Theory
A `Seq-feat` (Sequence Feature) attaches biological meaning to a specific sequence interval. In Biopython, `record.features` is a list of `SeqFeature` objects. Each feature has:
- `.type`: The biological classification (e.g. `'source'`, `'gene'`, `'CDS'`, `'exon'`, `'misc_feature'`)
- `.location`: Sequence coordinates specifying `start`, `end`, and `strand`
- `.qualifiers`: A dictionary of key-value pairs (e.g. `{'gene': ['HBB']}`, `{'protein_id': ['NP_000509.1']}`)


In [ ]:
# Worked Example: Inspecting Seq-feat items on HBB
print(f"Total Seq-feat annotations: {len(record.features)}")

# Tabulate feature types present
feature_types = {}
for feat in record.features:
    feature_types[feat.type] = feature_types.get(feat.type, 0) + 1

print("\nFeature types breakdown:")
for ftype, count in feature_types.items():
    print(f"  {ftype:15} : {count}")

# Inspect the CDS (Coding Sequence) feature
cds_feat = next(f for f in record.features if f.type == "CDS")
print("\n=== CDS Seq-feat Details ===")
print("Location:   ", cds_feat.location)
print("Coordinates:", f"start={cds_feat.location.start}, end={cds_feat.location.end}, strand={cds_feat.location.strand}")
print("Gene symbol:", cds_feat.qualifiers.get("gene", ["None"])[0])
print("Protein ID: ", cds_feat.qualifiers.get("protein_id", ["None"])[0])
print("Translation (first 30 aa):", cds_feat.qualifiers.get("translation", [""])[0][:30])


In [ ]:
# Try it: Locate the 'exon' features and print their coordinates
exons = [f for f in record.features if f.type == "exon"]
print(f"HBB mRNA contains {len(exons)} exon annotations:")
for i, ex in enumerate(exons, 1):
    print(f"  Exon {i}: {ex.location}")


## 6. Compare with NCBI Web Interface
When viewing `NM_000518.5` on the NCBI website:
- At the top right, clicking **Display Settings** allows toggling between **GenBank** flat file and **ASN.1 (text)**.
- In the ASN.1 view, you can see the literal `Bioseq ::= { id { other { accession "NM_000518" , version 5 } } ... }` syntax!
Biopython parses this exact underlying object graph into `record.id`, `record.annotations`, and `record.features`.


## 7. Summary
- The NCBI Data Model formalizes sequence records into objects rather than raw text flat files.
- `Seq-id` identifies the molecule; `Bioseq` holds sequence data; `Seq-descr` provides contextual metadata; `Seq-feat` annotates specific sequence intervals.
- In Biopython:
  - `record.id` = `Seq-id`
  - `record.annotations` = `Seq-descr`
  - `record.features` = `Seq-feat`

| NCBI Model Concept | Biopython Attribute | Description |
|---|---|---|
| `Seq-id` | `record.id` | Accession and version |
| `Bioseq` | `record.seq`, `len(record)` | Molecule data, length, sequence |
| `Seq-descr` | `record.description`, `record.annotations` | Organism, lineage, citations, comments |
| `Seq-feat` | `record.features` | Genes, CDS, exons, variations with locations |
| `Qualifiers` | `feature.qualifiers` | Dictionary of feature tags (`gene`, `product`, `protein_id`) |


## 8. Next Steps
- Apply your understanding in `U03_P_NCBI_Data_Model.ipynb`.
- Self-test data model vocabulary in `U03_Q_NCBI_Data_Model.ipynb`.
